In [ ]:
%%capture
!pip install -r requirements.txt
from utils import *

wd = WarpDrive()

business_segment = wd.get_args("business_segment")

In [ ]:
# Function to read a dataset in parquet
def read_parquet_from_azure(container, directory):
    """
    Reads a parquet file from object storage and returns a DataFrame.
 
    Parameters:
        container : container / bucket name
        directory : path to the parquet file
 
    Returns:
        pd.DataFrame
    """
    with wd.store.get_object_readable_stream(container, directory) as stream:
        data = pd.read_parquet(stream)
 
    return data

# Function to extract the window details from the pargs json
def extract_windows(pargs):
    window_defitions = []

    for window in pargs.get("windows", {}).get("definition", []):
        for window_name, window_values in window.items():
            window_defitions.append({
                "window": window_name,
                "start_date": window_values.get("start_date"),
                "end_date": window_values.get("end_date")
            })

    return window_defitions

In [ ]:
# Read the pipeline configuration arguments
file_path = 'anomaly_detection/retail_banking_new/data'
customers_dir = os.path.join(file_path, 'customers.parquet')
accounts_dir = os.path.join(file_path, 'accounts.parquet')
transactions_dir = os.path.join(file_path, 'transactions.parquet')

pargs = json.loads(wd.store.read_object('nimbus-uno-usbank', 'anomaly_detection/retail_banking/arguments/pipeline_arguments.json').decode('utf-8'))
 
container = pargs['paths']['container']
prefix_base = pargs['paths']['prefix_base']
total_num_windows = pargs['windows']['count']
start_date = pargs['data_period']['start_date']
end_date = pargs['data_period']['end_date']

# Apply the extract_windows function to fetch the windows info from json file
windows_info = extract_windows(pargs)
 
# Read customers data, accounts and transactions data
customers_data = read_parquet_from_azure(container, customers_dir)
accounts_data = read_parquet_from_azure(container, accounts_dir)
transactions_data = read_parquet_from_azure(container, transactions_dir)

In [ ]:
# Merge Customer & Account data
accounts_data = accounts_data.rename(columns={'linked_customer_id' : 'customer_id'})
customers_accounts_merged_data = pd.merge(customers_data, accounts_data, on='customer_id', how='left', indicator=True)

customers_accounts_merged_data = customers_accounts_merged_data.drop('_merge', axis=1, errors='ignore')

In [ ]:
# Merge Customer-Account-Transaction data
tms_data = pd.merge(transactions_data, customers_accounts_merged_data, on=['account_id', 'customer_id'], how='left', indicator=True)

tms_data = tms_data.drop('_merge', axis=1, errors='ignore')

tms_data["transaction_datetime"] = pd.to_datetime(tms_data["transaction_datetime"])   

In [ ]:
overall_summary_data = pd.DataFrame({
    "Business Segment": [business_segment, business_segment],
    "Metric": ["Total Records", "Total Features"],
    "Value": [
        f"{tms_data.shape[0]:,}",
        f"{tms_data.shape[1]:,}"
    ]
})

columns_data = pd.DataFrame({
    "Business Segment": business_segment,
    "Data": "Transformed",
    "Features": tms_data.columns
})

wd.save_table(overall_summary_data, f"Transformed Data Summary - {business_segment}")
wd.save_table(columns_data, f"Transformed Data Features Summary - {business_segment}")

In [ ]:
# Rolling window logic to extract multiple time-based subsets of TMS data
def extract_tms_subsets(tms_data, windows_info):
    windowed_data = {}

    for window in windows_info:
        window_name = window["window"]
        start_date = pd.to_datetime(window["start_date"])
        end_date = pd.to_datetime(window["end_date"])

        filtered_data = tms_data[
            (tms_data["transaction_datetime"] >= start_date) &
            (tms_data["transaction_datetime"] <= end_date)
        ].copy()

        windowed_data[window_name] = filtered_data

    return windowed_data
    
# Apply rolling window logic to extract TMS data for different windows
windowed_data = extract_tms_subsets(tms_data, windows_info)

# Create a summary of the window level data 
windows_info_dict = {w["window"]: w for w in windows_info}

rows = []

for window, data in windowed_data.items():
    rows.append({
        "Window": window,
        "Start Date": windows_info_dict[window]["start_date"],
        "End Date": windows_info_dict[window]["end_date"],
        "#Records": data.shape[0],
        "#Columns": data.shape[1]
    })

window_summary_data = pd.DataFrame(rows)
wd.save_table(window_summary_data, f"Rolling Window Data Summary - {business_segment}")

# Extract the test dataset and save as parquet
training_end_date = pd.to_datetime(windows_info[-1]["end_date"])
test_tms_data = tms_data.loc[tms_data["transaction_datetime"] > training_end_date].copy()
test_tms_data.shape

In [ ]:
# Function to write the data to Azure Blob
def save_df_to_azure(data: pd.DataFrame, container_name: str, file_path: str, output_type: str):
    buffer = io.BytesIO()
    if output_type == 'parquet':
        data.to_parquet(buffer, index=False)
    elif output_type == 'csv':
        data.to_csv(buffer, index=False)
    elif output_type == 'excel':
        data.to_excel(buffer, index=False)        
    buffer.seek(0)
    wd.store.put_object(container_name=container_name, key=file_path, data=buffer)
    print(f"DataFrame saved to Nimbus Azure Blob Container: {container_name}/{file_path}")

In [ ]:
date_folder_fmt = '_'.join([start_date.replace('-', ''), end_date.replace('-', '')])

# Save training rolling window data as parquet
for window, data in windowed_data.items():
    output_file_path = os.path.join(file_path, date_folder_fmt, f'tms_data_{window}.parquet')
    save_df_to_azure(data, container, output_file_path, 'parquet')

# Save testing rolling window data as parquet
output_file_path = os.path.join(file_path, date_folder_fmt, 'tms_data_test.parquet')
save_df_to_azure(test_tms_data, container, output_file_path, 'parquet')

# Save full merged TMS data for use in downstream feature engineering scripts
output_file_path = os.path.join(file_path, date_folder_fmt, 'tms_data_full.parquet')
save_df_to_azure(tms_data, container, output_file_path, 'parquet')